# Cognitive History

Long-running agents accumulate extensive execution history. Without management, this history would eventually overflow the LLM's context window. The framework's `CognitiveHistory` implements a three-tier memory system that automatically compresses older memories while keeping recent details readily available.

In this tutorial, we'll build a **multi-step data collection agent** and observe how the memory tiers work: recent steps stay in full detail (Working Memory), older steps become summaries (Short-term Memory), and the oldest steps get compressed into a concise paragraph (Long-term Memory).

## Initialize

First, let's set up the LLM and define our data collection tools.

In [ ]:
import os
from openai import AsyncOpenAI

api_key = os.environ.get("OPENAI_API_KEY", "your-api-key")
base_url = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")

In [ ]:
from bridgic.amphibious import AmphibiousAutoma, CognitiveContext, CognitiveWorker, think_unit
from bridgic.model import OpenAILlm

llm = OpenAILlm(
    model="gpt-4o-mini",
    api_key=api_key,
    base_url=base_url,
)

In [ ]:
from bridgic.core import tool

@tool(description="Scrape data from a web page")
async def scrape_page(url: str) -> str:
    return f"Scraped data from {url}: 150 records found, 12 fields per record"

@tool(description="Clean and validate scraped data")
async def clean_data(source: str, rules: str) -> str:
    return f"Cleaned data from {source}: removed 12 duplicates, fixed 5 format issues"

@tool(description="Store processed data in the database")
async def store_data(table: str, record_count: int) -> str:
    return f"Stored {record_count} records in table '{table}'"

@tool(description="Generate a progress report for the data collection task")
async def progress_report(completed: int, total: int) -> str:
    return f"Progress: {completed}/{total} sources processed ({completed/total*100:.0f}%)"

## The Three-Tier Memory Model

`CognitiveHistory` organizes agent execution history into three tiers, each with a different level of detail:

| Tier | Content | LLM Visibility |
|---|---|---|
| **Working Memory** | Most recent N steps | Full details — tool calls, arguments, return values, everything |
| **Short-term Memory** | Next M older steps | One-line summaries; full details available on demand via Acquiring |
| **Long-term Memory** | Oldest steps | LLM-compressed paragraph; individual step details no longer available |

- **Working Memory** is the agent's immediate context. The LLM sees every detail of recent steps, so it can reason precisely about what just happened.
- **Short-term Memory** trades detail for capacity. Steps are stored as summaries, but the agent can "look back" and request full details when needed (via the Acquiring policy).
- **Long-term Memory** is the most compressed tier. When Short-term Memory overflows, the LLM compresses the oldest summaries into a single paragraph. Individual step access is no longer possible, but the gist is preserved.

## CognitiveHistory Configuration

You configure the memory tiers by setting three parameters when creating a `CognitiveHistory` instance:

In [ ]:
from bridgic.amphibious import CognitiveHistory

# Default configuration
history = CognitiveHistory(
    working_memory_size=5,    # Keep last 5 steps in full detail
    short_term_size=20,       # Keep next 20 steps as summaries
    compress_threshold=10,    # Trigger LLM compression after 10 pending steps
)

# For long-running agents, you might increase these:
large_history = CognitiveHistory(
    working_memory_size=10,   # More recent detail
    short_term_size=50,       # Larger summary buffer
    compress_threshold=20,    # Less frequent compression
)

## Watching Memory Tiers in Action

Let's build a data collection agent that processes multiple sources. With 5 sources and 4 steps per source (scrape, clean, store, report), the agent will execute ~20 steps — enough to see all three memory tiers at work.

In [ ]:
class DataCollector(AmphibiousAutoma[CognitiveContext]):
    collector = think_unit(
        CognitiveWorker.inline(
            "Process the next data source: scrape, clean, and store the data. "
            "Then report progress. Work through sources one at a time."
        ),
        max_attempts=15,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.collector

In [ ]:
agent = DataCollector(llm=llm)
result = await agent.arun(
    goal=(
        "Collect data from 5 sources: "
        "site-a.com/data, site-b.com/data, site-c.com/data, "
        "site-d.com/data, site-e.com/data. "
        "For each: scrape, clean, store, and report progress."
    ),
    tools=[scrape_page, clean_data, store_data, progress_report],
)
print(result)

## Inspecting the Memory State

After a long run, we can inspect what each memory tier contains. The `CognitiveHistory` tracks how many steps live in each tier and what information is available.

In [ ]:
# After running the agent, we can inspect the cognitive history
# The context tracks how many steps are in each tier

# CognitiveHistory summary shows the memory structure:
# - Long-term: compressed summary paragraph
# - Short-term: one-line summaries (details available via Acquiring)
# - Working: full step details

## Acquiring + Cognitive History

When the agent needs to recall details from an older step (one that has been moved to Short-term Memory), it uses the **Acquiring** policy. Here's how it works:

1. The LLM sees a summary like *"Step 5: Scraped data from site-c.com"*
2. It requests details: `details: [{field: "cognitive_history", index: 5}]`
3. The framework reveals the full step (tool calls, arguments, results)
4. The LLM re-thinks with the additional information

This means the agent **never permanently loses access** to Short-term Memory — it can always "look back" when needed, without keeping everything in the prompt at all times. The key insight is that most steps don't need to be revisited, so keeping them as summaries saves significant context window space.

## Memory Migration Flow

As the agent accumulates steps, memory automatically migrates through the tiers:

```
New step added
    |
    v
[Working Memory] --- latest N steps (full detail)
    |
    v (overflow)
[Short-term Memory] --- next M steps (summaries, details on demand)
    |
    v (overflow, triggers LLM compression)
[Long-term Memory] --- compressed paragraph (no individual step access)
```

The migration is automatic and transparent:

- When a new step is added and Working Memory is full, the oldest working step moves to Short-term Memory (compressed to a summary).
- When Short-term Memory exceeds `compress_threshold` pending steps, the oldest batch is compressed by the LLM into the Long-term Memory paragraph.
- The agent keeps running without interruption — it doesn't need to know about the compression happening behind the scenes.

## Custom History Configuration

You can customize the memory tiers by creating a custom context with your own `CognitiveHistory` defaults:

In [ ]:
from pydantic import Field, ConfigDict

class CustomContext(CognitiveContext):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    # Override the default CognitiveHistory with custom sizes
    cognitive_history: CognitiveHistory = Field(
        default_factory=lambda: CognitiveHistory(
            working_memory_size=8,
            short_term_size=30,
            compress_threshold=15,
        )
    )

<div style="text-align: center; margin: 2rem 0;">
<hr style="border: none; border-top: 2px solid #e2e8f0;">
</div>

## What have we learnt?

In this tutorial, we explored the framework's automatic memory management:

- **CognitiveHistory** implements a three-tier memory system: Working Memory (full detail), Short-term Memory (summaries with on-demand details), and Long-term Memory (LLM-compressed paragraph).
- Memory tiers are configured via `working_memory_size`, `short_term_size`, and `compress_threshold`.
- The framework **automatically migrates** steps from Working → Short-term → Long-term as the agent accumulates history.
- **Acquiring + Short-term Memory**: The LLM can always request full details of any step in Short-term Memory via the Acquiring policy — nothing is permanently lost until Long-term compression.
- This system prevents context window overflow in long-running agents while preserving the ability to recall important details.

The three-tier memory is what makes Bridgic Amphibious agents capable of sustained, long-running tasks without losing track of what they've done.